In [ ]:
"""
FedOpt Implementation for Wheat Plant Diseases Dataset
Dataset: https://www.kaggle.com/datasets/kushagra3204/wheat-plant-diseases
Settings:
  - 5 clients, Dirichlet non-IID (alpha=0.5)
  - ResNet18, SGD lr=0.001 momentum=0.9, batch=32
  - 5 local epochs, 10 communication rounds
  - Server-side Adam optimizer (lr=0.001)
  - Saves: per-round CSV, final JSON, confusion matrix text
"""

import os
import json
import copy
import csv
import time
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Config ───────────────────────────────────────────────────────────────────
NUM_CLIENTS      = 5
ALPHA            = 0.5          # Dirichlet concentration
NUM_ROUNDS       = 50
LOCAL_EPOCHS     = 5
BATCH_SIZE       = 32
LOCAL_LR         = 0.001        # client SGD lr
MOMENTUM         = 0.9
SERVER_LR        = 0.001        # FedOpt Adam lr (server)
BETA1            = 0.9          # Adam β₁
BETA2            = 0.999        # Adam β₂
EPS_ADAM         = 1e-8         # Adam ε
OUTPUT_DIR       = "/kaggle/working/fedopt_wheat_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ── Dataset path detection ────────────────────────────────────────────────────
def find_dataset_root():
    candidates = [
        "/kaggle/input/datasets/kushagra3204/wheat-plant-diseases/data",
        "/kaggle/input/wheat-plant-diseases/Wheat Plant Diseases",
        "/kaggle/input/wheat-plant-diseases/wheat-plant-diseases",
    ]
    for path in candidates:
        if os.path.isdir(path):
            # Verify it contains image subfolders
            subdirs = [d for d in os.listdir(path)
                       if os.path.isdir(os.path.join(path, d))]
            if len(subdirs) >= 2:
                print(f"Dataset root: {path}  ({len(subdirs)} classes)")
                return path
            # Go one level deeper if subdirs are present but only 1
            for sub in subdirs:
                deeper = os.path.join(path, sub)
                inner = [d for d in os.listdir(deeper)
                         if os.path.isdir(os.path.join(deeper, d))]
                if len(inner) >= 2:
                    print(f"Dataset root: {deeper}  ({len(inner)} classes)")
                    return deeper
    raise RuntimeError("Dataset not found. Attach the Kaggle dataset and retry.")

DATA_ROOT = find_dataset_root()

# ── Transforms ───────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# ── Full dataset (train transform for client splits) ─────────────────────────
full_dataset = datasets.ImageFolder(DATA_ROOT, transform=train_transform)
NUM_CLASSES  = len(full_dataset.classes)
print(f"Classes ({NUM_CLASSES}): {full_dataset.classes}")

# ── Dirichlet non-IID partitioning ───────────────────────────────────────────
def dirichlet_split(dataset, num_clients, alpha, seed=SEED):
    rng = np.random.default_rng(seed)
    labels = np.array(dataset.targets)
    num_classes = len(dataset.classes)
    client_indices = [[] for _ in range(num_clients)]

    for cls in range(num_classes):
        cls_idx = np.where(labels == cls)[0]
        rng.shuffle(cls_idx)
        proportions = rng.dirichlet(alpha * np.ones(num_clients))
        proportions = (proportions * len(cls_idx)).astype(int)
        # Fix rounding to ensure all indices are used
        diff = len(cls_idx) - proportions.sum()
        proportions[0] += diff
        start = 0
        for c, cnt in enumerate(proportions):
            client_indices[c].extend(cls_idx[start:start + cnt].tolist())
            start += cnt
    return client_indices

client_indices = dirichlet_split(full_dataset, NUM_CLIENTS, ALPHA)
for i, idx in enumerate(client_indices):
    print(f"  Client {i}: {len(idx)} samples")

# ── Test set (eval transform on same ImageFolder) ────────────────────────────
eval_dataset = datasets.ImageFolder(DATA_ROOT, transform=eval_transform)
all_indices  = list(range(len(eval_dataset)))
# Hold out last 20% as test (reproducible with fixed order)
rng_split = np.random.default_rng(SEED + 1)
test_indices = rng_split.choice(all_indices, size=int(0.2 * len(all_indices)), replace=False).tolist()
test_loader  = DataLoader(Subset(eval_dataset, test_indices),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ── Model builder ─────────────────────────────────────────────────────────────
def build_model():
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model  # stored on CPU; moved to DEVICE during compute

# ── Evaluation helper ─────────────────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    model.to(DEVICE)
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            preds = model(imgs).argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    model.to("cpu")

    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
    rec  = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
    f1   = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
    cm   = confusion_matrix(all_labels, all_preds)
    return acc, prec, rec, f1, cm, all_labels, all_preds

# ── Local training (FedOpt client step) ──────────────────────────────────────
def local_train(global_state_dict, client_idx_list):
    """Train a copy of the global model locally; return delta (Δw = w_local - w_global)."""
    model = build_model()
    model.load_state_dict(copy.deepcopy(global_state_dict))
    model.to(DEVICE)

    subset  = Subset(full_dataset, client_idx_list)
    loader  = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=2, pin_memory=True)
    optimizer = torch.optim.SGD(model.parameters(), lr=LOCAL_LR, momentum=MOMENTUM)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for _ in range(LOCAL_EPOCHS):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()

    local_state  = {k: v.cpu() for k, v in model.state_dict().items()}
    global_state = global_state_dict

    # Δw = w_local - w_global
    delta = {k: local_state[k] - global_state[k] for k in global_state}
    n_samples = len(client_idx_list)
    return delta, n_samples

# ── FedOpt server aggregation (Adam) ─────────────────────────────────────────
class FedOptServer:
    def __init__(self, global_model):
        # m_t and v_t (first/second moment estimates) stored on CPU
        self.m = {k: torch.zeros_like(v) for k, v in global_model.state_dict().items()}
        self.v = {k: torch.zeros_like(v) for k, v in global_model.state_dict().items()}
        self.t = 0  # time step

    def aggregate(self, global_state_dict, deltas, sample_counts):
        """
        Weighted pseudo-gradient = weighted average of client deltas.
        Server applies Adam update: w ← w + lr * m̂ / (√v̂ + ε)
        """
        self.t += 1
        total = sum(sample_counts)
        weights = [n / total for n in sample_counts]

        # Weighted average delta (pseudo-gradient)
        pseudo_grad = {}
        for k in global_state_dict:
            pg = sum(w * d[k] for w, d in zip(weights, deltas))
            pseudo_grad[k] = pg

        new_state = {}
        for k in global_state_dict:
            g = pseudo_grad[k]
            # Bias-corrected Adam
            self.m[k] = BETA1 * self.m[k] + (1 - BETA1) * g
            self.v[k] = BETA2 * self.v[k] + (1 - BETA2) * g * g
            m_hat = self.m[k] / (1 - BETA1 ** self.t)
            v_hat = self.v[k] / (1 - BETA2 ** self.t)
            new_state[k] = global_state_dict[k] + SERVER_LR * m_hat / (torch.sqrt(v_hat) + EPS_ADAM)
        return new_state

# ── Save helpers ──────────────────────────────────────────────────────────────
csv_path = os.path.join(OUTPUT_DIR, "fedopt_wheat_per_round.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Round", "Accuracy", "Precision", "Recall", "F1_Score", "Time_sec"])

def save_round(rnd, acc, prec, rec, f1, elapsed):
    with open(csv_path, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([rnd, f"{acc:.4f}", f"{prec:.4f}", f"{rec:.4f}", f"{f1:.4f}", f"{elapsed:.1f}"])
    print(f"  [Round {rnd:02d}] Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  F1={f1:.4f}  ({elapsed:.1f}s)")

def save_confusion_matrix(cm, class_names, rnd, final=False):
    tag  = "FINAL" if final else f"round_{rnd:02d}"
    path = os.path.join(OUTPUT_DIR, f"confusion_matrix_{tag}.txt")
    max_name = max(len(c) for c in class_names)
    col_w    = max(max_name, 5)

    with open(path, "w") as f:
        f.write(f"=== Confusion Matrix — FedOpt / WheatPatNet  [{tag}] ===\n\n")
        header = " " * (max_name + 2) + "  ".join(f"{c:>{col_w}}" for c in class_names)
        f.write(header + "\n")
        f.write("-" * len(header) + "\n")
        for i, row in enumerate(cm):
            row_str = f"{class_names[i]:<{max_name}}  " + "  ".join(f"{v:>{col_w}}" for v in row)
            f.write(row_str + "\n")
        f.write("\nRow = Actual, Col = Predicted\n")
    return path

def save_final_json(results_list, class_names, best_round):
    best = results_list[best_round - 1]
    summary = {
        "algorithm":   "FedOpt",
        "dataset":     "WheatPatNet",
        "num_clients": NUM_CLIENTS,
        "alpha":       ALPHA,
        "num_rounds":  NUM_ROUNDS,
        "local_epochs": LOCAL_EPOCHS,
        "server_optimizer": "Adam",
        "server_lr":   SERVER_LR,
        "local_lr":    LOCAL_LR,
        "batch_size":  BATCH_SIZE,
        "num_classes": NUM_CLASSES,
        "classes":     class_names,
        "best_round":  best_round,
        "best_accuracy":  best["accuracy"],
        "best_precision": best["precision"],
        "best_recall":    best["recall"],
        "best_f1":        best["f1"],
        "per_round_results": results_list,
    }
    path = os.path.join(OUTPUT_DIR, "fedopt_wheat_summary.json")
    with open(path, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\nSummary JSON saved → {path}")

# ── Main FedOpt loop ──────────────────────────────────────────────────────────
def main():
    print("\n" + "=" * 60)
    print("  FedOpt — Wheat Plant Diseases Dataset")
    print("=" * 60)

    global_model  = build_model()   # lives on CPU between rounds
    global_state  = {k: v.cpu() for k, v in global_model.state_dict().items()}
    server        = FedOptServer(global_model)
    class_names   = full_dataset.classes

    all_results = []
    best_f1, best_round = -1.0, 1

    for rnd in range(1, NUM_ROUNDS + 1):
        t0 = time.time()
        print(f"\n[Round {rnd}/{NUM_ROUNDS}] — client training …")

        deltas, counts = [], []
        for c in range(NUM_CLIENTS):
            delta, n = local_train(global_state, client_indices[c])
            deltas.append(delta)
            counts.append(n)

        # Server Adam update
        global_state = server.aggregate(global_state, deltas, counts)

        # Evaluate
        global_model.load_state_dict(global_state)
        acc, prec, rec, f1, cm, _, _ = evaluate(global_model, test_loader)
        elapsed = time.time() - t0

        save_round(rnd, acc, prec, rec, f1, elapsed)
        save_confusion_matrix(cm, class_names, rnd)

        result = {
            "round": rnd,
            "accuracy":  round(float(acc),  4),
            "precision": round(float(prec), 4),
            "recall":    round(float(rec),  4),
            "f1":        round(float(f1),   4),
            "time_sec":  round(elapsed, 1),
        }
        all_results.append(result)

        if f1 > best_f1:
            best_f1, best_round = f1, rnd
            # Save best model weights
            torch.save(global_state, os.path.join(OUTPUT_DIR, "fedopt_wheat_best_model.pt"))

    # Final confusion matrix
    global_model.load_state_dict(global_state)
    _, _, _, _, cm_final, _, _ = evaluate(global_model, test_loader)
    save_confusion_matrix(cm_final, class_names, NUM_ROUNDS, final=True)
    save_final_json(all_results, class_names, best_round)

    print("\n" + "=" * 60)
    print(f"  Best Round : {best_round}  |  Best F1 : {best_f1:.4f}")
    print(f"  Results saved to: {OUTPUT_DIR}")
    print("=" * 60)

if __name__ == "__main__":
    main()